In [12]:
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [13]:
# Load trained model
model = load_model("lstm_rnn_hamlet.h5")
print("✅ Model loaded")

# Load tokenizer
with open("tokenizer.pickle", "rb") as handle:
    tokenizer = pickle.load(handle)
print("✅ Tokenizer loaded")

# Show sample from tokenizer
print("Sample word_index:", dict(list(tokenizer.word_index.items())[:10]))

✅ Model loaded
✅ Tokenizer loaded
Sample word_index: {'the': 1, 'and': 2, 'to': 3, 'of': 4, 'i': 5, 'you': 6, 'a': 7, 'my': 8, 'it': 9, 'in': 10}


In [14]:
# Vocabulary size
total_words = len(tokenizer.word_index) + 1
print("Total words in vocab:", total_words)

# Same max sequence length used during training
max_sequence_len = 14   # <- replace with your training value
print("Max sequence length (training):", max_sequence_len)

Total words in vocab: 4818
Max sequence length (training): 14


In [15]:
def predict_next_word(seed_text):
    """Generate next word(s) given a seed sentence."""
    print("\nSeed text:", seed_text)

    # Step A: Convert to tokens
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    print(f"\n[{1}] Tokenized sequence:", token_list)

    # Step B: Pad to match training length
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    print(f"[{2}] After padding {token_list} and length is {len(token_list[0])}", )

    # Step C: Predict probabilities
    predicted_probs = model.predict(token_list, verbose=0)
    print(f"[{3}] Predicted probabilities: {predicted_probs}, Shape: {len(predicted_probs[0])}", )

    # Step D: Find most probable word index
    predicted_index = np.argmax(predicted_probs, axis=1)[0]
    print(f"[{4}] Predicted word index:", predicted_index)

    # Step E: Convert index back to word
    predicted_word = None
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            predicted_word = word
            break
    print(f"[{5}] Predicted word:", predicted_word)

    # Append predicted word
    seed_text += " " + predicted_word

    return seed_text

In [16]:
seed = "But looke, the Morne in Russet"
result = predict_next_word(seed)
print("\n✅ Final Generated Text:", result)


Seed text: But looke, the Morne in Russet

[1] Tokenized sequence: [19, 141, 1, 1224, 10, 2021]
[2] After padding [[   0    0    0    0    0    0    0   19  141    1 1224   10 2021]] and length is 13
[3] Predicted probabilities: [[2.3360171e-16 4.0879713e-11 8.2426010e-11 ... 2.8372437e-31
  1.6133560e-28 2.3957604e-16]], Shape: 4818
[4] Predicted word index: 2022
[5] Predicted word: mantle

✅ Final Generated Text: But looke, the Morne in Russet mantle


In [17]:
seed = "When yond same Starre that's"
result = predict_next_word(seed)
print("\n✅ Final Generated Text:", result)


Seed text: When yond same Starre that's

[1] Tokenized sequence: [95, 1910, 275, 693, 249]
[2] After padding [[   0    0    0    0    0    0    0    0   95 1910  275  693  249]] and length is 13
[3] Predicted probabilities: [[1.5643842e-15 1.2431410e-08 2.8991581e-15 ... 1.5832239e-22
  7.2548642e-22 1.6445092e-15]], Shape: 4818
[4] Predicted word index: 1911
[5] Predicted word: westward

✅ Final Generated Text: When yond same Starre that's westward
